# Chaîne complète SGG — collecter, traiter, archiver, expédier

Ce carnet prend une tranche de l'index des décrets du SGG et la mène jusqu'à une archive déposée sur votre VPS.

## Pourquoi Colab plutôt que le VPS

Le VPS a deux limites que Colab n'a pas : **19 Go de disque** partagés avec cinq autres projets, et un hébergeur qui bride le processeur au-delà d'un certain usage — il l'a déjà fait une fois. Colab offre ~100 Go éphémères, plusieurs cœurs, et une bande passante qui rend la collecte rapide.

Le VPS redevient ce qu'il fait bien : **stocker et servir**.

## Le cycle

```
  Colab                                         VPS                Téléphone
  ─────                                         ───                ─────────
  collecte pages N..M  ──┐
  traitement BLDP        ├─→  archive + sources  ──→  stockage  ──→  rsync
  empaquetage          ──┘         (espace vérifié avant envoi)
```

**Une tranche à la fois.** Le rapatriement vers le téléphone est lent ; la collecte suivante n'a de sens qu'une fois la place libérée.

## Avant de lancer

Ce carnet ne s'ouvre pas dans un navigateur : il est exécuté depuis votre terminal par le CLI Colab, qui n'a besoin d'aucun onglet.

```bash
colab new    -s bldp
colab upload -s bldp ~/.ssh/colab_bldp /content/colab_bldp
colab exec   -s bldp -f notebooks/collecte_traitement_sgg.ipynb
colab stop   -s bldp
```

**Ne demandez pas de GPU.** Mesuré le 6 septembre 2026 : le runtime `--gpu T4` donne les mêmes 2 cœurs et les mêmes 12 Go de RAM que le runtime CPU, sur un processeur plus ancien et avec 23 Go de disque en moins. Aucun de nos outils — Tesseract, Ghostscript, PyMuPDF — n'a de version CUDA : la carte resterait inutilisée, en consommant un quota bien plus rare. Le GPU deviendra utile le jour où les embeddings seront activés, pas avant.

Réglez ensuite **`PAGE_DEBUT` et `PAGE_FIN`** dans la cellule 2 : c'est ce qui définit la tranche.

## 1. Installer l'outillage

Tesseract et Ghostscript pour l'OCR, OCRmyPDF pour les piloter, puis BLDP depuis le dépôt public.

Chaque étape vérifie son code de retour et affiche sa sortie d'erreur. Une installation qui échoue en silence coûte une heure de recherche trois cellules plus loin — c'est déjà arrivé ici.

> Le clone est anonyme. Si le dépôt redevenait privé, la cellule demanderait un jeton GitHub (portée `repo`) sans l'afficher ni le laisser dans `.git/config`.


In [ ]:
import subprocess, sys
from pathlib import Path

DEPOT = Path("/content/bldp")
URL = "github.com/ekacel1/BLDP.git"

def executer(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)


def courir(cmd, cwd=None, titre=None):
    """Lance une commande, montre sa sortie au fil de l'eau, et LEVE si elle echoue.

    « !commande » dans un carnet ne fait ni l'un ni l'autre : un echec n'y
    leve pas, et le carnet enchaine — on empaquette alors un corpus incomplet
    en annoncant un succes. Filtrer par « | tail » aggrave le tout, puisque le
    code de retour observe devient celui de tail.
    """
    if titre:
        print(titre, flush=True)
    processus = subprocess.Popen(
        [str(a) for a in cmd], cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace",
    )
    for ligne in processus.stdout:
        sys.stdout.write(ligne)
        sys.stdout.flush()
    processus.wait()
    if processus.returncode != 0:
        raise RuntimeError(
            "echec (code %d) : %s" % (processus.returncode, " ".join(str(a) for a in cmd)[:200])
        )

# -- binaires systeme -----------------------------------------------------
print("installation des binaires...", flush=True)
# Colab livre Tesseract 4.1.1. Le corpus existant a ete oceriser avec la
# 5.4 : deux moteurs sur un meme corpus produiraient des textes qui ne se
# comparent pas. On prend donc la 5 via le PPA, et on le DIT si on echoue.
executer(["apt-get", "-qq", "update"])
executer(["apt-get", "-qq", "install", "-y", "software-properties-common"])
ppa = executer(["add-apt-repository", "-y", "ppa:alex-p/tesseract-ocr5"])
if ppa.returncode == 0:
    executer(["apt-get", "-qq", "update"])
else:
    print("  PPA Tesseract 5 indisponible — on restera sur la version de base.")

r = executer(["apt-get", "-qq", "install", "-y", "tesseract-ocr",
              "tesseract-ocr-fra", "ghostscript", "poppler-utils", "rsync"])
if r.returncode != 0:
    print(r.stderr[-800:])
    raise RuntimeError("apt a echoue — voir ci-dessus.")

# -- le depot -------------------------------------------------------------
if (DEPOT / ".git").exists():
    r = executer(["git", "-C", str(DEPOT), "pull", "--ff-only"])
    print("depot :", (r.stdout or r.stderr).strip().splitlines()[-1] if (r.stdout or r.stderr).strip() else "a jour")
else:
    r = executer(["git", "clone", "--depth", "1", f"https://{URL}", str(DEPOT)])
    if r.returncode != 0:
        # Aucun repli interactif : « colab exec » n'a personne au clavier, et
        # un getpass y attendrait indefiniment. Le depot est public ; si le
        # clone echoue, c'est le reseau ou l'URL, et il faut le voir tout de
        # suite plutot que de rester bloque.
        print((r.stderr or r.stdout)[-800:])
        raise RuntimeError("clone impossible — voir ci-dessus.")
    print("depot : clone")

# -- BLDP et ses dependances d'OCR ---------------------------------------
# « pip install -e . » seul ne pose que PyMuPDF et PyYAML : OCRmyPDF,
# pytesseract et Pillow sont un extra. Sans « [ocr] », le pipeline echoue
# a la premiere page a oceriser — plusieurs heures apres le lancement.
print("installation de BLDP...", flush=True)
r = executer([sys.executable, "-m", "pip", "-q", "install", "-e", ".[ocr]"], cwd=DEPOT)
if r.returncode != 0:
    print((r.stderr or r.stdout)[-1200:])
    raise RuntimeError("pip a echoue — voir ci-dessus.")

# -- verification ---------------------------------------------------------
print()
manques = []
for outil in ("tesseract", "gs", "ocrmypdf", "rsync"):
    r = executer([outil, "--version"])
    if r.returncode != 0:
        manques.append(f"{outil} absent")
    else:
        print(f"{outil:<12}: {(r.stdout or r.stderr).splitlines()[0]}")

langues = executer(["tesseract", "--list-langs"]).stdout
print("langues     :", " ".join(langues.split()[1:]))
v = executer(["tesseract", "--version"])
ligne = (v.stdout or v.stderr).splitlines()[0]
if "tesseract 5" not in ligne:
    print()
    print("  ATTENTION : " + ligne + " — le corpus existant a ete produit")
    print("  avec la 5.4. Les textes de cette tranche ne seront pas")
    print("  strictement comparables a ceux des lots 1 et 2.")

if "fra" not in langues:
    manques.append("le francais manque a Tesseract")

# Un premier essai rate laisse parfois un « bldp » vide en cache : Python
# a vu le dossier du depot avant qu il contienne quoi que ce soit, et en a
# fait un paquet-espace-de-noms. L import suivant rend ce fantome sans
# jamais relire le disque. On purge, et on verifie d ou vient le module.
import importlib
for nom in [n for n in sys.modules if n == "bldp" or n.startswith("bldp.")]:
    del sys.modules[nom]
importlib.invalidate_caches()
sys.path.insert(0, str(DEPOT))
try:
    import bldp
    origine = getattr(bldp, "__file__", None)
    if origine is None or not origine.startswith(str(DEPOT)):
        manques.append("bldp importe depuis " + str(origine) + " au lieu de " + str(DEPOT))
    else:
        print("bldp        :", bldp.__version__, "(" + origine + ")")
except Exception as exc:
    manques.append("bldp : " + str(exc))

if manques:
    print()
    for m in manques:
        print("  ECHEC :", m)
    raise RuntimeError("Outillage incomplet — inutile de continuer.")
print()
print("Tout est en place.")

## 2. Définir la tranche

L'index des décrets compte **1 619 pages de 20 documents**, de 2026 (page 1) à 1998 et au-delà. Les pages **1 à 306** sont déjà collectées et traitées : le corpus actuel s'arrête à `decret-2013-275`.

Une tranche de 250 pages représente ~5 000 documents et ~9 Go — le maximum que le VPS peut accueillir aujourd'hui.

### Ce que Colab autorise, et ce qui contraint vraiment

La liste des usages interdits ne vise rien de ce qu'on fait ici : ni torrent, ni proxy, ni minage, ni cassage de mots de passe. Deux points méritent quand même d'être nommés.

Colab interdit « l'hébergement de fichiers, la diffusion de médias ou tout service web étranger au calcul interactif ». Télécharger des PDF pour les **traiter** est du calcul ; les faire seulement transiter vers un serveur ne le serait pas. Ici les documents sont océrisés, découpés en articles et structurés sur place — le transit des sources est un effet de bord du traitement, pas sa finalité. C'est du bon côté de la ligne, mais la ligne existe.

L'interdiction du « contrôle à distance : shells SSH, bureaux distants » vise le fait d'ouvrir un accès **vers** la machine Colab. Nous faisons l'inverse : une connexion sortante vers votre serveur. Personne ne pilote le runtime depuis l'extérieur.

**La vraie contrainte est ailleurs.** Ce carnet n'est plus destiné à un navigateur : il est lancé depuis votre terminal par le CLI Colab, en quatre commandes.

```bash
colab new    -s bldp
colab upload -s bldp ~/.ssh/colab_bldp /content/colab_bldp
colab exec   -s bldp -f notebooks/collecte_traitement_sgg.ipynb
colab stop   -s bldp
```

Il n'y a donc plus d'onglet à garder ouvert — mais **votre portable doit rester allumé et connecté** : c'est lui qui héberge le démon qui maintient la VM en vie. Restent la limite de 12 h par session, et le fait qu'une VM gratuite est fournie « au mieux », sans garantie d'allocation ni de continuité.

Corollaire décisif : **aucune cellule ne peut poser de question.** Rien ne lit l'entrée standard, rien n'attend de saisie. Tout ce dont le carnet a besoin doit être en place avant qu'il démarre — d'où le `colab upload` de la clé, ci-dessous.

Ressources mesurées sur une VM gratuite, le 6 septembre 2026 : **2 cœurs, 12 Go de RAM, 88 Go de disque**, Tesseract 4.1.1 préinstallé, ni Ghostscript ni OCRmyPDF. Le runtime `--gpu T4` donne les deux mêmes cœurs sur un processeur plus ancien et 23 Go de disque en moins : sans emploi ici, aucun de nos outils n'ayant de version CUDA. Il le deviendra le jour où les embeddings seront activés.

In [ ]:
PAGE_DEBUT = 332      # premiere page NON collectee
PAGE_FIN = 481        # incluse — 150 pages
NOM_LOT = "lot4"

VPS_HOTE = "191.96.1.191"
VPS_UTILISATEUR = "root"
CONTACT = "dikdokmoney@gmail.com"   # part dans le User-Agent

# Politesse envers le SGG : une seconde entre deux requetes. Ce n'est pas
# negociable — c'est ce qui evite de se faire bloquer, et ce qui est correct.
DELAI_SECONDES = 1.0

# --- Mesures du lot 3 (438 documents, 25 pages, 6 septembre 2026) -----------
# Elles remplacent les estimations d'origine, qui tablaient sur 20 documents
# par page et 1,93 Mo par document. On garde les chiffres mesures tant qu'une
# tranche n'en fournit pas de meilleurs.
DOCS_PAR_PAGE = 17.5          # 438 documents sur 25 pages
MO_PAR_DOCUMENT = 1.72        # 754 Mo de sources pour 438 documents
KO_ARCHIVE_PAR_DOCUMENT = 24.5
SECONDES_PAR_DOCUMENT = 1.1   # sur 2 fils, VM Colab gratuite

pages = PAGE_FIN - PAGE_DEBUT + 1
documents = pages * DOCS_PAR_PAGE
sources_go = documents * MO_PAR_DOCUMENT / 1024
archive_mo = documents * KO_ARCHIVE_PAR_DOCUMENT / 1024

collecte_h = pages * (DOCS_PAR_PAGE + 1) * DELAI_SECONDES / 3600
traitement_h = documents * SECONDES_PAR_DOCUMENT / 3600

print(f"tranche      : pages {PAGE_DEBUT} a {PAGE_FIN}  ({pages} pages)")
print(f"documents    : ~{documents:.0f}")
print(f"sources      : ~{sources_go:.1f} Go  (a stocker sur le VPS, puis a rapatrier)")
print(f"archive      : ~{archive_mo:.0f} Mo")
print()
print(f"collecte     : ~{collecte_h:.1f} h  (dont l'essentiel en simple politesse)")
print(f"traitement   : ~{traitement_h:.1f} h")
print(f"cycle total  : ~{collecte_h + traitement_h:.1f} h")
print()

# Deux limites, et ce n'est plus le calcul qui borne.
if collecte_h + traitement_h > 10:
    print("  TROP LONG pour une session (12 h max). Reduisez PAGE_FIN.")
if sources_go > 11:
    print("  TROP GROS pour le VPS : ~11 Go utilisables une fois la marge gardee.")
    print(f"  Pages tenables : ~{int(11 * 1024 / (DOCS_PAR_PAGE * MO_PAR_DOCUMENT))}")
print("Si la session est coupee en route, relancez : le carnet reprend ou il")
print("s'etait arrete (les PDF deja presents et valides sont sautes).")

## 3. Vérifier la place sur le VPS — avant de collecter

Collecter des gigaoctets pour découvrir ensuite qu'ils ne rentrent pas serait du temps perdu. On demande d'abord.

La clé SSH doit **déjà être sur la VM** quand le carnet démarre : en exécution pilotée par `colab exec`, aucune cellule ne peut réclamer de saisie. Elle y est déposée par une commande, avant le lancement :

```bash
colab upload -s bldp ~/.ssh/colab_bldp /content/colab_bldp
```

Le fichier passe tel quel, octet pour octet. C'est ce qui règle définitivement le `error in libcrypto` rencontré plus tôt : ce message venait d'un champ de saisie qui écrasait les sept lignes de la clé en une seule. Il n'y a plus de champ de saisie.

> Prenez `colab_bldp`, **sans** `.pub` — le `.pub` est la moitié publique, déjà sur le serveur. Cette clé n'ouvre que ce serveur : ni tunnel, ni shell, ni GitHub.

In [ ]:
import pathlib, stat, subprocess

DEPOSE = pathlib.Path("/content/colab_bldp")   # ce que « colab upload » a mis
CLE = pathlib.Path("/root/.ssh/id_ed25519")
CLE.parent.mkdir(parents=True, exist_ok=True)


def cle_valide(chemin):
    """La cle se laisse-t-elle lire par ssh-keygen ?

    Verifier les bornes BEGIN/END ne suffit pas : une cle tronquee les garde
    et echoue plus tard, ailleurs, sur un « error in libcrypto » qui ne dit
    pas d ou il vient.
    """
    r = subprocess.run(["ssh-keygen", "-y", "-f", str(chemin)],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stderr or "").strip()


if CLE.exists() and not cle_valide(CLE)[0]:
    print("La cle deja en place est illisible — on la remplace.")
    CLE.unlink()

if not CLE.exists():
    if not DEPOSE.exists():
        print("Cle absente. Depuis votre terminal, avant de relancer :")
        print()
        print("    colab upload -s bldp ~/.ssh/colab_bldp /content/colab_bldp")
        print()
        raise RuntimeError("Aucune cle en " + str(DEPOSE) + " — voir ci-dessus.")

    donnees = DEPOSE.read_bytes()
    if donnees.lstrip().startswith(b"ssh-"):
        raise RuntimeError(
            "C est la cle PUBLIQUE. Reprenez le fichier de meme nom, sans .pub."
        )
    # Un fichier venu de Windows peut arriver en CRLF ; OpenSSH n en veut pas.
    donnees = donnees.replace(bytes([13, 10]), bytes([10])).replace(bytes([13]), bytes([10]))
    if not donnees.endswith(bytes([10])):
        donnees += bytes([10])
    CLE.write_bytes(donnees)
    CLE.chmod(stat.S_IRUSR | stat.S_IWUSR)

    bonne, motif = cle_valide(CLE)
    if not bonne:
        CLE.unlink()
        raise RuntimeError("Cle illisible : " + motif[:200])
    print("cle acceptee")

SSH = ["ssh", "-i", str(CLE), "-o", "IdentitiesOnly=yes",
       "-o", "StrictHostKeyChecking=accept-new",
       "-o", "BatchMode=yes", "-o", "ConnectTimeout=20",
       VPS_UTILISATEUR + "@" + VPS_HOTE]

r = subprocess.run(SSH + ["df -BG --output=avail / | tail -1"],
                   capture_output=True, text=True, timeout=60)
if r.returncode != 0:
    print(r.stderr.strip()[:400])
    raise RuntimeError("Connexion au VPS impossible — voir ci-dessus.")

libre_go = int("".join(c for c in r.stdout if c.isdigit()))
besoin_go = pages * 20 * 1.93 / 1024 + 0.5
print("VPS : " + str(libre_go) + " Go libres")
print("tranche : ~%.1f Go necessaires (sources + archive)" % besoin_go)
print()
if libre_go < besoin_go + 8:
    print("  ATTENTION : la marge est insuffisante.")
    print("  Rapatriez et effacez une tranche precedente, ou reduisez PAGE_FIN.")
    print("  Pages tenables : environ %d" % int((libre_go - 8) * 1024 / (20 * 1.93)))
    raise RuntimeError("Place insuffisante sur le VPS — voir ci-dessus.")
print("  Place suffisante — on peut collecter.")

## 4. Collecter la tranche

On lit l'index page par page, on en extrait les métadonnées — titre officiel, numéro, date de publication, description — puis on télécharge chaque PDF.

**Un document déjà connu n'est pas retéléchargé.** La reprise est gratuite : relancer la cellule après une coupure ne refait que ce qui manque.

In [ ]:
import hashlib, html, json, re, time, urllib.request
from pathlib import Path

BASE = "https://sgg.gouv.bj"
AGENT = f"BLDP/0.1 (collecte juridique; +{CONTACT})"
RACINE = Path(f"/content/{NOM_LOT}")
PDFS = RACINE / "input" / "decrets"
PDFS.mkdir(parents=True, exist_ok=True)

# Motifs repris du greffon sgg-benin, pour extraire les memes champs.
ENTREE = re.compile(r"<aside class='doc")
LIEN = re.compile(r"doc/([a-z0-9%\-]+)/", re.I)
TITRE = re.compile(r"class='doc-title'[^>]*>([^<]+)<", re.I)
NUMERO = re.compile(r"<i class='num'>([^<]+)</i>", re.I)
PUBLIE = re.compile(r"Publi[ée] le\s*([0-9]{2})\.([0-9]{2})\.([0-9]{4})")
DESCR = re.compile(r"class='doc-desc'[^>]*>(.*?)</", re.I | re.S)

def lire(url, essais=3):
    for n in range(essais):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": AGENT})
            with urllib.request.urlopen(req, timeout=60) as rep:
                return rep.read()
        except Exception as exc:
            if n == essais - 1:
                raise
            time.sleep(2 ** n)
    return b""

def url_page(p):
    return f"{BASE}/documentheque/decrets/" if p == 1 else f"{BASE}/documentheque/decrets/{p}/"

fiches, vus = [], set()
for page in range(PAGE_DEBUT, PAGE_FIN + 1):
    try:
        contenu = lire(url_page(page)).decode("utf-8", "replace")
    except Exception as exc:
        print(f"  page {page} illisible : {exc}")
        continue
    blocs = ENTREE.split(contenu)[1:]
    for bloc in blocs:
        m = LIEN.search(bloc)
        if m is None:
            continue
        slug = m.group(1)
        if slug in vus:
            continue
        vus.add(slug)
        d = PUBLIE.search(bloc)
        t = TITRE.search(bloc)
        n = NUMERO.search(bloc)
        desc = DESCR.search(bloc)
        fiches.append({
            "slug": slug,
            "url": f"{BASE}/doc/{slug}/",
            "titre": html.unescape(t.group(1)).strip() if t else None,
            "numero": html.unescape(n.group(1)).strip() if n else None,
            "publieLe": f"{d.group(3)}-{d.group(2)}-{d.group(1)}" if d else None,
            "description": re.sub(r"<[^>]+>", " ", html.unescape(desc.group(1))).strip()[:600] if desc else None,
            "categorie": "decret",
            "page": page,
        })
    if (page - PAGE_DEBUT) % 25 == 0:
        print(f"  page {page} — {len(fiches)} fiches", flush=True)
    time.sleep(DELAI_SECONDES)

print()
print(f"{len(fiches)} fiches recensees sur {PAGE_FIN - PAGE_DEBUT + 1} pages")
(RACINE / "fiches.json").write_text(json.dumps(fiches, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
# Telechargement des PDF. Chaque fichier est verifie : un PDF commence par
# « %PDF ». Le SGG sert parfois autre chose — un script PHP a deja ete
# rencontre sous une URL de decret. Ce qui n'est pas un PDF est ecarte, pas
# stocke : un corpus juridique n'accueille pas ce qu'il n'a pas identifie.
recus, rejetes, deja = 0, [], 0
for i, f in enumerate(fiches, 1):
    cible = PDFS / f"{f['slug'].replace('%', '_')}.pdf"
    if cible.exists() and cible.stat().st_size > 1000:
        # Meme sur la reprise, on reverifie l'en-tete : un fichier deja la
        # n'est pas une preuve qu'il est un PDF. Le §33 vaut aussi pour ce
        # qu'on croit deja acquis.
        if cible.read_bytes()[:4] != b"%PDF":
            cible.unlink()
        else:
            f["fichier"] = cible.name
            f["octets"] = cible.stat().st_size
            f["empreinte"] = hashlib.sha256(cible.read_bytes()).hexdigest()
            deja += 1
            continue
    try:
        contenu = lire(f"{BASE}/doc/{f['slug']}/download")
    except Exception as exc:
        rejetes.append((f["slug"], f"telechargement : {exc}"))
        continue
    if not contenu.startswith(b"%PDF"):
        rejetes.append((f["slug"], f"pas un PDF ({contenu[:12]!r})"))
        continue
    cible.write_bytes(contenu)
    f["fichier"] = cible.name
    f["octets"] = len(contenu)
    f["empreinte"] = hashlib.sha256(contenu).hexdigest()
    recus += 1
    if i % 100 == 0:
        print(f"  {i}/{len(fiches)} — {recus} recus, {len(rejetes)} rejetes", flush=True)
    time.sleep(DELAI_SECONDES)

fiches = [f for f in fiches if "fichier" in f]
(RACINE / "fiches.json").write_text(json.dumps(fiches, ensure_ascii=False, indent=1), encoding="utf-8")

poids = sum(f["octets"] for f in fiches)
print()
print(f"{len(fiches)} PDF disponibles ({poids/1073741824:.2f} Go) — {recus} nouveaux, {deja} deja la")
if rejetes:
    print(f"{len(rejetes)} rejete(s) :")
    for slug, motif in rejetes[:10]:
        print(f"   {slug} : {motif}")

## 4 bis. Mettre les sources à l'abri — tout de suite

`/content/` est éphémère. Une session Colab qui tombe à la troisième heure emporte tout ce qui a été téléchargé : plusieurs gigaoctets et une heure de requêtes polies au SGG, à refaire.

Les sources partent donc **avant** le traitement, pas après. Une coupure ne coûte alors que du calcul — jamais la collecte.


In [ ]:
poids_sources = sum(f["octets"] for f in fiches) / 1073741824
r = subprocess.run(SSH + ["df -BG --output=avail / | tail -1"],
                   capture_output=True, text=True, timeout=60)
libre = int("".join(c for c in r.stdout if c.isdigit()))
print(f"VPS : {libre} Go libres — sources a envoyer : {poids_sources:.2f} Go")

if libre - poids_sources < 6:
    raise RuntimeError(
        f"Place insuffisante : {libre} Go pour {poids_sources:.1f} Go de sources, "
        "marge de 6 Go exigee pour l'archive a venir. Rapatriez et effacez une "
        "tranche precedente avant de continuer."
    )

subprocess.run(SSH + [f"mkdir -p /opt/bldp/archives /opt/bldp/{NOM_LOT}/input/decrets"], timeout=60)
TRANSPORT = f"ssh -i {CLE} -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ServerAliveInterval=15"

courir(["rsync", "-a", "--partial", "--info=progress2", "-e", TRANSPORT,
        str(PDFS) + "/",
        f"{VPS_UTILISATEUR}@{VPS_HOTE}:/opt/bldp/{NOM_LOT}/input/decrets/"],
       titre="Envoi des sources vers le VPS...")

n_local = len(list(PDFS.glob("*.pdf")))
r = subprocess.run(SSH + [f"ls -1 /opt/bldp/{NOM_LOT}/input/decrets/*.pdf 2>/dev/null | wc -l"],
                   capture_output=True, text=True, timeout=120)
distant = r.stdout.strip()
print()
print(f"sources : {n_local} ici, {distant} sur le VPS")
if distant != str(n_local):
    # On leve : ce compte est la seule garantie que le traitement peut tomber
    # sans rien couter. Continuer avec des sources incompletes reviendrait a
    # produire une archive qu'on ne pourrait plus reconstituer.
    raise RuntimeError(
        f"Le compte ne tombe pas juste ({n_local} ici, {distant} la-bas) — "
        "relancez cette cellule avant de traiter."
    )
print("  Les sources sont a l'abri. Le traitement peut tomber sans rien couter.")

## 5. Bâtir le catalogue

BLDP sait confronter ce qu'il lit dans un PDF à ce que le portail annonce — c'est ce qui remplit `source_url`, confirme les numéros et signale les écarts. Cette confrontation attend un index au format du collecteur LCF.

On le fabrique ici avec les métadonnées déjà recueillies : le code de confrontation reste **exactement celui du pipeline**, déjà éprouvé.

In [ ]:
import sqlite3

LCF = RACINE / "catalogue"
(LCF / "index").mkdir(parents=True, exist_ok=True)
db_path = LCF / "index" / "lcf.db"
if db_path.exists():
    db_path.unlink()

db = sqlite3.connect(db_path)
db.executescript("""
CREATE TABLE documents (document_id TEXT PRIMARY KEY, source_id TEXT, native_id TEXT,
    canonical_url TEXT, current_version INTEGER, version_count INTEGER, status TEXT);
CREATE TABLE document_versions (document_id TEXT, version_no INTEGER, content_hash TEXT,
    fetched_at TEXT);
CREATE TABLE content_objects (content_hash TEXT PRIMARY KEY, byte_size INTEGER,
    mime_type TEXT, detected_mime TEXT, storage_path TEXT, verify_status TEXT);
CREATE TABLE document_metadata (document_id TEXT, version_no INTEGER, raw_json TEXT,
    common_json TEXT, provenance_json TEXT);
CREATE VIEW v_current_documents AS
SELECT d.document_id, d.source_id, d.native_id, d.canonical_url, d.status,
       v.version_no, v.fetched_at, v.content_hash, c.byte_size, c.mime_type, c.verify_status
FROM documents d
JOIN document_versions v ON v.document_id = d.document_id AND v.version_no = d.current_version
JOIN content_objects c ON c.content_hash = v.content_hash;
""")

import datetime
maintenant = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="milliseconds")

for f in fiches:
    interne = hashlib.sha256(f"bj.sgg.decrets/{f['slug']}".encode()).hexdigest()
    empreinte = f"sha256:{f['empreinte']}"
    # Le chemin est relatif au dossier de donnees, comme chez LCF.
    chemin = f"input/decrets/{f['fichier']}"
    db.execute("INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?,?,?)",
               (interne, "bj.sgg.decrets", f["slug"], f["url"], 1, 1, "stored"))
    db.execute("INSERT INTO document_versions VALUES (?,?,?,?)",
               (interne, 1, empreinte, maintenant))
    db.execute("INSERT OR REPLACE INTO content_objects VALUES (?,?,?,?,?,?)",
               (empreinte, f["octets"], "application/pdf", "application/pdf", chemin, "ok"))
    brut = {k: f[k] for k in ("titre", "numero", "publieLe", "description", "categorie") if f.get(k)}
    commun = {"authority": "Secrétariat Général du Gouvernement", "documentKind": "decret",
              "language": "fr", "reference": f.get("numero") or "", "issuedAt": f.get("publieLe") or ""}
    prov = [{"field": c, "at": f["url"], "locator": f"index page {f['page']}"}
            for c in brut]
    db.execute("INSERT INTO document_metadata VALUES (?,?,?,?,?)",
               (interne, 1, json.dumps(brut, ensure_ascii=False),
                json.dumps(commun, ensure_ascii=False), json.dumps(prov, ensure_ascii=False)))
db.commit()
db.close()

# Le magasin d'objets de LCF pointe vers les PDF : un lien physique suffit,
# et ne coute aucun octet.
import os
for f in fiches:
    lien = LCF / "input" / "decrets" / f["fichier"]
    lien.parent.mkdir(parents=True, exist_ok=True)
    if not lien.exists():
        try:
            os.link(PDFS / f["fichier"], lien)
        except OSError:
            lien.symlink_to(PDFS / f["fichier"])

sys.path.insert(0, "/content/bldp")
from bldp.core.crawl import LcfIndex
with LcfIndex(LCF) as index:
    print(f"catalogue : {index.count()} fiches")
    ex = next(index.records(), None)
    if ex:
        print(f"exemple   : {ex.document_id} | {ex.number} | {ex.url}")
        print(f"            PDF resolu : {ex.content_path.exists()}")

## 6. Traiter

Le pipeline complet : classification, extraction ou OCR, nettoyage, découpage en articles, métadonnées confrontées au catalogue, relations, qualité.

Le découpage par tranches de 100 est indispensable — un lot entier en mémoire a déjà fait tuer un traitement de six mille documents après quatre heures.

In [ ]:
import multiprocessing, yaml

coeurs = multiprocessing.cpu_count()
# Garder un coeur « pour le systeme » est sain sur une grosse machine et
# absurde sur deux : cela revient a n'en utiliser qu'un seul. Or le travail
# lourd (ocrmypdf, tesseract, gs) se fait dans des sous-processus, et les
# fils Python passent leur temps a les attendre, GIL relache — deux fils
# sur deux coeurs se recouvrent donc reellement. Une VM Colab gratuite en
# a exactement deux. Le pipeline force de lui-meme « ocr.jobs = 1 » des que
# workers > 1, pour eviter N processus x M fils.
fils = coeurs if coeurs <= 2 else coeurs - 1
config = {
    "paths": {k: str(RACINE / "data" / k) for k in
              ("raw", "processed", "validated", "embeddings", "exports", "traites")},
    "ingest": {"copy_to_raw": False},
    "pipeline": {"resume": True, "workers": fils, "slice_size": 100},
    "ocr": {"keep_sidecar_for": "review"},
    "export": {"include_page_text": True, "by_category": True},
    "embeddings": {"enabled": False},
    "vectorstore": {"enabled": False},
    "crawl": {"enabled": True, "index": str(LCF)},
    "privacy": {"allow_external_calls": False},
}
config["paths"]["input"] = str(RACINE / "input")
config["paths"]["logs"] = str(RACINE / "logs")
config["paths"]["database"] = str(RACINE / "data" / "exports" / "legal_database.sqlite")

chemin_config = RACINE / "config.yaml"
chemin_config.write_text(yaml.safe_dump(config, allow_unicode=True), encoding="utf-8")
print(f"{coeurs} coeurs — {fils} fils, tranches de 100")
print(f"config : {chemin_config}")

In [ ]:
import time

debut = time.time()
courir([sys.executable, "-m", "bldp", "pipeline", str(RACINE / "input"),
        "--config", str(chemin_config)],
       cwd=str(DEPOT),
       titre="Traitement en cours — la sortie defile au fil de l'eau.")
duree = time.time() - debut

# Ce chiffre est le seul qui permette de dimensionner les tranches suivantes :
# tout ce qui precede n'etait qu'estimation, faite sur une autre machine.
traites = len(list(PDFS.glob("*.pdf")))
print()
print(f"Traitement termine en {duree / 60:.0f} min pour {traites} document(s).")
if traites:
    par_doc = duree / traites
    print(f"debit mesure : {par_doc:.1f} s/document sur {fils} fil(s)")
    print(f"une page d'index (~20 documents) coute ~{par_doc * 20 / 60:.0f} min")
    print()
    tenables = int((10 * 3600) / (par_doc * 20 + 21 * DELAI_SECONDES))
    print(f"En restant sous 10 h de session, une tranche peut aller jusqu'a")
    print(f"~{tenables} pages. C'est ce chiffre qui fixe PAGE_FIN la prochaine fois.")

In [ ]:
base = RACINE / "data" / "exports" / "legal_database.sqlite"
d = sqlite3.connect(f"file:{base}?mode=ro", uri=True)
q = lambda s: d.execute(s).fetchone()[0]

print(f"{q('SELECT COUNT(*) FROM documents')} documents | "
      f"{q('SELECT COUNT(*) FROM articles')} articles | "
      f"{q('SELECT COUNT(*) FROM pages')} pages")
print()

# Un champ vide n'est pas un echec en soi : le pipeline prefere le vide a
# l'invention. Mais un taux qui grimpe signale une source qui a change de
# forme — c'est ce qu'on veut voir avant d'archiver.
for champ in ("source_url", "number", "date", "authority", "title"):
    requete = (
        "SELECT SUM(CASE WHEN {0} IS NULL OR {0} = '' THEN 1 ELSE 0 END) "
        "FROM documents"
    ).format(champ)
    print(f"  {champ:<12} absent : {q(requete)}")

print()
print("  sans aucun article :",
      q("SELECT COUNT(*) FROM documents d WHERE NOT EXISTS "
        "(SELECT 1 FROM articles a WHERE a.document_id = d.document_id)"))
d.close()

## 7. Empaqueter

L'archive porte le corpus traité **et** le manifeste de reconstitution : pour chaque document, son URL d'origine et l'empreinte du fichier traité. C'est ce qui rend l'effacement des sources réversible.

In [ ]:
ARCHIVES = RACINE / "archives"
courir([sys.executable, "scripts/empaqueter_lot.py", NOM_LOT,
        "--exports", str(RACINE / "data"), "--lcf", str(LCF),
        "--sortie", str(ARCHIVES)],
       cwd=str(DEPOT), titre="Empaquetage du lot...")

## 8. Expédier l'archive

Les sources sont déjà sur le VPS depuis l'étape 4 bis. Il ne reste que l'archive — le produit du travail.

Son empreinte est vérifiée des deux côtés : c'est elle qui autorisera, plus tard, à effacer quoi que ce soit.


In [ ]:
archive = sorted(ARCHIVES.glob(f"{NOM_LOT}-corpus-*.zip"))[-1]
poids = archive.stat().st_size / 1073741824

r = subprocess.run(SSH + ["df -BG --output=avail / | tail -1"],
                   capture_output=True, text=True, timeout=60)
libre = int("".join(c for c in r.stdout if c.isdigit()))
print(f"VPS : {libre} Go libres — archive : {poids:.2f} Go")
if libre - poids < 2:
    raise RuntimeError(f"Place insuffisante pour l'archive ({libre} Go restants).")

courir(["rsync", "-a", "--partial", "--info=progress2", "-e", TRANSPORT,
        str(archive),
        f"{VPS_UTILISATEUR}@{VPS_HOTE}:/opt/bldp/archives/"],
       titre="Envoi de l'archive...")

In [ ]:
# L'empreinte doit survivre au transfert. Tant qu'elle n'est pas confirmee
# des deux cotes, rien ne doit etre efface nulle part.
local = hashlib.sha256(archive.read_bytes()).hexdigest()
r = subprocess.run(SSH + [f"sha256sum /opt/bldp/archives/{archive.name} | cut -d' ' -f1"],
                   capture_output=True, text=True, timeout=300)
distant = r.stdout.strip()
print(f"Colab : {local}")
print(f"VPS   : {distant}")
print()
if local != distant:
    # On leve. C'est ce verrou qui autorisera plus tard a effacer les sources :
    # le laisser passer en simple avertissement rendrait l'effacement aveugle.
    raise RuntimeError(
        "Les empreintes different — le transfert est incomplet ou corrompu. "
        "Ne rien effacer, relancer la cellule precedente."
    )
print("  IDENTIQUE — notez cette empreinte, elle sert au rapatriement.")

r = subprocess.run(SSH + ["df -h / | tail -1"], capture_output=True, text=True, timeout=60)
print()
print(f"VPS apres envoi : {r.stdout.split()[3]} libres")

## 9. La suite, dans l'ordre

**Deux choses reviennent sur le téléphone : l'archive *et* les sources.** Pas
l'une ou l'autre. L'archive est le produit du travail ; les sources sont les
originaux, et rien d'autre ne les contient — l'archive porte le corpus traité
et le manifeste qui permet de les retrouver, pas les PDF eux-mêmes.

Sur votre téléphone, sous Termux :

```bash
termux-wake-lock
tmux new -s dl        # Ctrl-b puis d pour detacher
```

**1. L'archive** — quelques dizaines de mégaoctets, rapide :

```bash
until rsync -avP root@191.96.1.191:/opt/bldp/archives/lot3-corpus-*.zip ~/legal-data/; do
  sleep 15
done
sha256sum ~/legal-data/lot3-corpus-*.zip     # comparer a l'empreinte affichee ci-dessus
```

**2. Les sources** — plusieurs centaines de mégaoctets, c'est ce qui prend du
temps et ce qui cadence le rythme des tranches :

```bash
until rsync -avP root@191.96.1.191:/opt/bldp/lot3/input/decrets/ ~/legal-data/lot3-sources/; do
  sleep 15
done
ls -1 ~/legal-data/lot3-sources/*.pdf | wc -l     # doit egaler le compte annonce plus haut
```

**3. Alors seulement**, libérez le VPS :

```bash
ssh root@191.96.1.191 'rm -rf /opt/bldp/lot3/input/decrets && df -h / | tail -1'
```

Puis revenez ici, avancez `PAGE_DEBUT` et `PAGE_FIN`, changez `NOM_LOT`,
poussez sur GitHub, et relancez.

> **L'ordre compte, et les deux vérifications aussi.** L'empreinte prouve que
> l'archive est intacte ; le compte de fichiers prouve que les sources le sont.
> Effacer sans l'un des deux, c'est détruire des originaux sur la foi d'un
> transfert qu'on n'a pas contrôlé. Et ne lancez pas la tranche suivante avant
> d'avoir libéré la place — le VPS n'a pas de marge.